In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from keras import backend

In [ ]:
path_to_model = '../../run.01/saving/cg_deeponet/SavedModel'
path_to_data = '../../../data/train/pts.csv'

In [ ]:
dim = {
    "inp": [2+27, 1],
    "out": [2, 3, 9, 27, 1]
}
cols = {
    'x': np.split(
        np.arange(np.sum(dim['inp'])),
        np.cumsum(dim['inp'])
    )[:-1],
    'y': np.split(
        np.sum(dim['inp']) + np.arange(np.sum(dim['out'])),
        np.cumsum(dim['out'])
    )[:-1]
}

In [ ]:
data = pd.read_csv(path_to_data).to_numpy()
data = tf.nest.map_structure(lambda i: np.take(data, list(i), axis=1), cols)
x, y = list(data.values())

In [ ]:
model = tf.keras.models.load_model(path_to_model)

In [ ]:
layers = []
for layer in model.layers:
    for node in layer._inbound_nodes:
        layers_in = [inbound[0] for inbound in node.iterate_inbound()]
        for layer_in in layers_in:
            if "pre_net" in layer_in.name:
                layers.append({
                    "prenet": layer_in,
                    "trunk": layer
                })

In [ ]:
models = []
for layer in layers:
    models.append({
        "prenet": backend.function(model.inputs, layer["prenet"].output),
        "trunk": layer["trunk"]
    })

In [ ]:
data = []
nb_pts = 50
for m, model in enumerate(models):
    yp = model["prenet"](x)
    mn, mx = np.amin(y), np.amax(y)
    mn = np.sign(mn)*np.absolute(mn)*3
    mx = np.sign(mx)*np.absolute(mx)*1.5
    xt = np.linspace(
        mn,
        mx,
        num=nb_pts*3 if m == 0 else nb_pts
    )
    xt = xt.reshape(-1,1)
    yt = model["trunk"](xt)
    bt = np.hstack([xt, yt])
    np.savetxt(f'{model["trunk"].name}.txt', bt)
    plt.plot(xt, yt)
    plt.show()
    plt.close